# Decision tree classifier tutorial

This notebook walks through **training** and **out-of-sample evaluation** of `mlpackage.supervised_learning.DecisionTreeClassifier` using the Iris dataset. Each section explains **why** a step matters before showing code.

## Prerequisites and learning goals

**You will**
- Load a labeled dataset (`X`, `y`) suitable for multiclass classification.
- Split data into **training** vs **held-out test** subsets so predictions on the test split are genuinely **not seen during training**.
- Fit a decision tree by entropy / information gain (formulas in **Step 3** below; the folder README is high level only).
- Generate predictions on the test set and report accuracy plus a compact error breakdown.
- Visualize two Iris features with a scatter plot and compare **`max_depth`** on the same split.

**Requirements** (already listed in repo `requirements.txt`): `numpy`, `scikit-learn` (for data + split utilities), `pandas` and `matplotlib` for tables and plotting.


In [ ]:
from __future__ import annotations

import numpy as np

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

from mlpackage.supervised_learning import DecisionTreeClassifier

## Step 1: Load Iris

The **UCI Iris** benchmark is commonly used for teaching supervised learning: three species (classes), four numeric measurements per flower. Features are comparable in scale, though trees do not strictly require normalization.

We retain **integer labels** `0`, `1`, `2`, which match how this classifier returns predictions.

In [ ]:
iris = load_iris()
X = np.asarray(iris.data)
y = np.asarray(iris.target)

feature_names = list(iris.feature_names)
target_names = list(iris.target_names)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Features:", feature_names)
print("Classes:", {i: target_names[i] for i in range(len(target_names))})
print("\nFirst rows of X:\n", X[:5])
print("\nLabel counts:", dict(zip(*np.unique(y, return_counts=True))))

## Step 2: Train / test split (out-of-sample holdout)

**Out-of-sample** means: evaluate on observations that **never appeared** in training. If we fit on full data then report accuracy on the same labels, estimates are unrealistically optimistic (data leakage).

**Stratification** preserves each class proportion in both splits—helpful when classes are mildly imbalanced. A fixed **`random_state`** ensures reproducibility for teaching and demos.

In [ ]:
RANDOM_STATE = 42
TEST_FRACTION = 0.30

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_FRACTION,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train:", X_train.shape[0], "samples")
print("Test:", X_test.shape[0], "samples")
print("Train label counts:", dict(zip(*np.unique(y_train, return_counts=True))))
print("Test label counts:", dict(zip(*np.unique(y_test, return_counts=True))))

## Step 3: Instantiate and fit the tree

This implementation grows a **binary** tree with rules `x_j <= t` vs `x_j > t`. Training picks splits to maximize **information gain** from Shannon entropy (base 2 in code).

**`max_depth`** caps recursion depth—smaller values often generalize better; `None` grows until purity, depth limit, or no improving split.

**`fit(X_train, y_train)`** uses only training rows.

### Entropy and class proportions

At a node $N$ with $n$ samples, class counts $n_k$, and proportions $p_k = n_k/n$ over classes $k \in \mathcal{K}$,

$$
H(N) = -\sum_{k \in \mathcal{K}} p_k \log_2 p_k,
$$

with the convention $0\log_2 0 = 0$. Pure nodes have $H=0$; entropy is larger when label mass is spread across classes.

### Information gain

A split on feature $j$ at threshold $t$ yields children $L$ (training points with $x_{ij} \le t$) and $R$ (with $x_{ij} > t$ in this implementation). With child sizes $n_L, n_R$,

$$
H_{\text{after}}(j,t) = \frac{n_L}{n} H(L) + \frac{n_R}{n} H(R).
$$

**Information gain** is the entropy drop:

$$
\mathrm{IG}(j,t) = H(N) - H_{\text{after}}(j,t).
$$

The code maximizes $\mathrm{IG}$ over all features and **distinct observed** thresholds on the training points in $N$.

### Leaves and greedy training

Each leaf predicts the **majority** class among training points in that cell. The classifier is **piecewise constant** on an axis-aligned partition of $\mathbb{R}^p$. Splits are **greedy** (locally optimal, not globally optimal smallest-error trees).



In [ ]:
MAX_DEPTH = 4

clf = DecisionTreeClassifier(max_depth=MAX_DEPTH)
clf.fit(X_train, y_train)

print("Fitted DecisionTreeClassifier with max_depth =", clf.max_depth)

## Step 4: Predict on held-out test data

**`predict(X_test)`** applies the learned thresholds along a path root-to-leaf for each sample. These predictions are comparable to **`y_test`** because `X_test` was excluded during `fit`.

In [ ]:
y_pred = clf.predict(X_test)

print("First 15 test predictions:", y_pred[:15])
print("First 15 true labels:     ", y_test[:15])

## Step 5: Evaluate accuracy and mistakes

**Test accuracy** is the fraction of correct labels on the held-out set:

$$
\mathrm{Acc} = \frac{1}{n}\sum_{i=1}^{n}\mathbf{1}\{\hat{y}_i = y_i\}.
$$

The classifier **`score(X, y)`** returns this mean for the `(X, y)` pairs you pass—here `(X_test, y_test)` for honest out-of-sample performance.



In [ ]:
test_accuracy = clf.score(X_test, y_test)
train_accuracy = clf.score(X_train, y_train)

print(f"Train accuracy: {train_accuracy:.3f}")
print(f"Test accuracy : {test_accuracy:.3f}")

In [ ]:
wrong = np.where(y_pred != y_test)[0]
print("Number of misclassified test samples:", wrong.size)

# Per-class correctness on the test fold
for klass in sorted(np.unique(y)):
    mask = y_test == klass
    hits = np.sum((y_pred == y_test) & mask)
    total = np.sum(mask)
    print(
        f"Class {klass} ({target_names[klass]}): {hits}/{total} correct"
    )

## Step 6: Inspect a handful of unseen rows

Connecting raw features to **`true`** vs **`predicted`** labels helps debug behavior on specific test examples.

In [ ]:
n_preview = min(12, X_test.shape[0])
preview_idx = slice(0, n_preview)

df = pd.DataFrame(X_test[preview_idx], columns=feature_names)
df["true_label"] = [target_names[v] for v in y_test[preview_idx]]
df["predicted"] = [target_names[v] for v in y_pred[preview_idx]]
display(df)

## Step 7: Visualize two Iris features (petal length vs petal width)

The fitted tree uses all four features together; here we plot **`petal length`** vs **`petal width`** (columns 2 and 3) to visualize how species cluster in two dimensions (not the full decision boundary in \(\mathbb{R}^4\), but helpful intuition).

In [ ]:
j0, j1 = 2, 3
colors = ["tab:orange", "tab:green", "tab:blue"]

plt.figure(figsize=(7, 5))
for klass in sorted(np.unique(y)):
    m = y == klass
    plt.scatter(
        X[m, j0],
        X[m, j1],
        c=colors[klass],
        label=target_names[klass],
        alpha=0.85,
        edgecolors="black",
        linewidths=0.3,
        s=40,
    )
plt.xlabel(feature_names[j0])
plt.ylabel(feature_names[j1])
plt.title("Iris samples (training + test plotted together for geometry)")
plt.legend()
plt.tight_layout()

# Also write a PNG so the figure exists outside interactive viewers (CI, chats, etc.)
from pathlib import Path

_nb_here = Path("decision_tree_classifier_tutorial.ipynb")
if _nb_here.is_file():
    _plot_path = _nb_here.with_name("iris_petal_scatter.png")
else:
    _plot_path = Path(
        "examples/supervised_learning/decision_tree_classifier/iris_petal_scatter.png"
    )
    _plot_path.parent.mkdir(parents=True, exist_ok=True)

plt.savefig(_plot_path, dpi=150, bbox_inches="tight")
print(f"Saved figure to: {_plot_path.resolve()}")
plt.show()

## Step 8: Compare depth regularization

Increasing **`max_depth`** can lift training accuracy but hurt test accuracy (overfitting). We refit several depths using the **same** `(X_train, X_test, …)` split (`RANDOM_STATE` fixed above) so comparisons are fair.

In [ ]:
depths_to_try = [1, 2, 4, None]
rows = []
for md in depths_to_try:
    mdl = DecisionTreeClassifier(max_depth=md)
    mdl.fit(X_train, y_train)
    rows.append(
        {
            "max_depth": "None" if md is None else md,
            "train_acc": mdl.score(X_train, y_train),
            "test_acc": mdl.score(X_test, y_test),
        }
    )

display(pd.DataFrame(rows))